# 1.5 SIMT 基础

本节介绍 AI Core SIMT 编程模型的核心概念。

---

## 学习目标

完成本节后，你将能够：
- 理解编程模型概述
- 理解 SIMT 抽象硬件架构
- 掌握 SIMT 线程架构
- 了解 SIMT 内存层级
- 掌握核函数的定义与调用
- 了解 SIMT 同步机制

---

## 概述

AI Core是AI处理器的计算核心，AI处理器通过多个AI Core实现并行计算。与传统CPU相比，AI处理器因其内部拥有更多的计算单元和相应的向量计算指令，更适合模型训练和推理场景，这使得单个硬件指令能完成多组数据的计算。AI处理器提供了以下两种编程模型：

- SIMD（Single Instruction Multiple Data）：单指令多数据。通过单条指令多个数据的方式实现并行计算。
- SIMT（Single Instruction Multiple Thread）：单指令多线程。通过单条指令多个线程的方式实现并行计算。

AI CPU是位于Device侧的处理器，具备与AI Core相同的内存访问能力，能够直接访问Device侧的内存资源；同时，它也可以像Host侧的CPU一样进行数据计算。

| 编程模型 | 计算空间 | 特点 |
|---------|---------|------|
| SIMD编程 | AI Core | 适合矩阵计算、连续计算的矢量算子及融合算子场景，提供SIMD与SIMT混合的高级编程方式。 |
| SIMT编程 | AI Core | 适用于离散访问场景、复杂分支控制场景。 |
| AI CPU编程 | AI CPU | 作为AI Core计算的补充。 |

**SIMT编程**：

适用于离散访问场景、矢量算子的复杂分支控制场景，也便于熟悉SIMT算子开发的人员快速掌握AI处理器上的算子开发。

---

## 1. 抽象硬件架构

AI Core上的SIMT（Single Instruction Multiple Thread，单指令多线程）编程允许指令对数据进行独立寻址，从而支持线程级并行计算。这种方式特别适用于离散访问和复杂控制逻辑等场景。

SIMT编程能够简化复杂算子和不规则控制流的开发，通过SIMT编程可以有效缓解包含分支等的发散计算，同时控制程序复杂度。在系统层面，这有助于提高硬件利用率和能效。如下图所示，AI处理器内部有多个Vector Core，每个Vector Core包含计算单元、Share Memory即Unified Buffer、寄存器和堆栈空间，核外的Global Memory是全局内存空间，被所有Vector Core共享。

<img src="./images/simt_abstract_hardware_architecture.png" alt="simt_abstract_hardware_architecture"  width="700px" >

以下是SIMT多线程计算涉及到的硬件资源的说明：

- 每个线程拥有独立的寄存器和栈空间，用于存储局部数据， 寄存器的数量受线程块内线程数量影响，线程数量越多，每个线程拥有的寄存器数量越少。
- Unified Buffer的一部分内存空间作为线程块内所有线程的共享内存（Share Memory），支持线程块内的线程进行数据交互，一部分作为读取Global Memory的Data Cache。
- SIMT模式中，读取Global Memory上的数据时，通过Data Cache单元完成数据中转，数据流经由Global Memory到Data Cache，再从Data Cache到寄存器。Data Cache是Unified Buffer中预留的一部分空间，其最大容量为128KB，最小容量为32KB，实际大小由用户自主分配。

---

## 2. 线程架构

SIMT编程模型的线程层次结构分为两层：

- 线程块网格（Grid）：由多个线程块（Thread Block）组成，使用内置变量gridDim来表示启用的线程块的个数，同一时刻一个AIV核只执行一个线程块任务。
- 线程块（Thread Block）：由若干线程（thread）组成，使用内置变量blockDim表示一个线程块启用的的线程个数，一个线程块最多可以启用2048个线程。

基于SIMT编程模型的程序，在AIV核上执行多个结构相同的线程块，执行的总线程数等于gridDim*blockDim。

<img src="./images/simt_thread_hierarchy.png" alt="simt_thread_hierarchy"  width="700px" >

gridDim由三维结构dim3来表示，{dimx，dimy，dimz}用于指定3个不同维度的线程块的大小，三维乘积的总数不超过65535，各线程块可通过线程块索引blockIdx进行标识。

**三维维度的含义**：
- **dimx**：通常表示数据宽度方向（width）或行方向，是最常用的维度。
- **dimy**：通常表示数据高度方向（height）或列方向，用于二维数据处理。
- **dimz**：通常表示数据深度方向（depth/channels）或批次方向，用于三维数据或多批次处理。

**典型应用场景**：
- 一维数据：只需设置dimx，dimy和dimz设为1。
- 二维数据（如矩阵）：dimx对应行数，dimy对应列数，dimz设为1。
- 三维数据（如图像）：dimx对应宽度，dimy对应高度，dimz对应通道数。
- 多批次处理：dimx和dimy处理单批次数据，dimz对应批次数量。

blockDim也由dim3三维结构表示，三维乘积的总数不超过2048，各线程可通过线程块内线程索引threadIdx进行标识。线程索引的计算示例如下图所示：

<img src="./images/simt_thread_index_calculation.png" alt="simt_thread_index_calculation"  width="700px" >

底层调度过程中，同一时刻一个AIV只能执行一个线程块任务，每个线程块会被切分成多个Warp依次调度并完成执行。Warp是执行相同指令的线程的集合，每个Warp包含32个线程。每个AIV核包含4个Warp调度器（Warp Scheduler），调度器编号scheduler id为warp id % 4。

---

## 3. 内存层级

SIMT线程可访问多种内存空间。下表汇总了SIMT编程中常见内存类型的作用域及其生命周期。

| 内存类型 | 线程作用域 | 生命周期 | 物理位置 |
|---------|-----------|---------|----------|
| 全局内存 | Grid | 应用程序 | Device |
| 共享内存 | Block | 核函数 | Vector Core |
| 栈 | Thread | 核函数 | Device |
| 寄存器 | Thread | 核函数 | Vector Core |

- 全局内存是所有线程均可直接访问的内存资源，即Global Memory；
- 共享内存是线程块内所有线程共享的内存，即Unified Buffer，生命周期和线程块一致。
- 每个线程独立的寄存器和栈，用于存储局部变量。

内存层级如下图所示。

<img src="./images/simt_memory_hierarchy.png" alt="simt_memory_hierarchy"  width="700px" >

#### 全局内存（Global Memory）

Device侧的全局内存是整个Grid中所有线程均可访问的内存空间，其作用与CPU系统中的随机存取存储器（RAM）相似，运行在Device侧的核函数可直接访问全局内存，这种方式与CPU上代码访问系统内存的方式相同。

全局内存具有持久性：通过全局内存分配的空间、其中存储的数据将持续保留，直到该内存空间被释放或应用程序终止。用户通过Runtime API完成Device侧全局内存的管理。Host使用aclrtMalloc分配Device侧的全局内存，并通过aclrtMemcpy将数据从Host拷贝到Device侧的全局内存，或将数据从Device的全局内存拷贝回Host内存；通过aclrtMalloc分配的Device全局内存需使用aclrtFree接口释放。有关Runtime API的更多信息与细节，可以参考《Runtime运行时 API》章节。

在实际开发过程中，用户需在核函数启动前，通过Runtime API完成全局内存的分配与初始化；在核函数执行期间，SIMT每个线程均可读取和写入数据到全局内存；核函数执行完毕后，其写入全局内存的结果可拷贝回Host。由于全局内存对Grid内所有线程均开放访问，因此必须严格规避线程间的数据竞争。

下述代码为全局内存的使用提供了简易示例。数组 x、y、z 均存储于全局内存中，通过以下核函数实现每个线程对全局内存的访问和存储。

```cpp
__global__ void add_custom(float* x, float* y, float* z, uint64_t total_length)
{
    // Calculate global thread ID
    int32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    // Maps to the row index of output tensor
    if (idx >= total_length) {
        return;
    }
    z[idx] = x[idx] + y[idx];
}
```

#### 共享内存（Unified Buffer）

共享内存是同一线程块内所有线程均可访问的内存空间，位于每个Vector Core（AIV）内部。与全局内存相比，共享内存的容量较小，但具有更高的带宽和更低的访问延迟，可视为内核执行期间由用户管理的高速缓存资源。由于共享内存可被线程块内的全部线程访问，因此需要注意避免同一线程块内线程间的数据竞争。通过使用asc_syncthreads接口，可以实现同一线程块内的线程同步，该函数会阻塞线程块内的所有线程，直至所有线程均执行到接口调用位置。

用户可通过动态或者静态方式申请共享内存。

1. 静态申请：分配一段指定大小的内存空间，其大小在编译时确定，不可动态修改，开发者通过数组分配申请使用。该方式将在后续版本中支持。

```cpp
__ubuf__ half staticBuf[1024];
```

2. 动态申请：用户需要通过<<<>>>中参数dynUBufSize指定动态内存的空间大小，其大小在运行期确定，SIMT编程中可通过以下方式申请使用动态内存。该方式将在后续版本中支持。

```cpp
extern __ubuf__ char dynamicBuf[];
```

由于Unified Buffer不仅作为共享内存，还有部分内存空间预留作内部使用，因此用户在申请共享内存时，应注意不能将所有共享内存用尽。如下图，Unified Buffer内存空间总大小为256KB，按功能划分为四个主要区域，从低地址到高地址依次为静态内存、动态内存、预留空间和Data Cache。

<img src="./images/simt_ub_memory_allocation.png" alt="simt_ub_memory_allocation"  width="700px" >

具体结构如下：

1. 静态内存和动态内存对用户静态、动态申请方式分配的内存。
2. 预留空间：编译器和Ascend C预留空间，大小固定为8KB。
3. Data Cache：SIMT专有的Data Cache空间，用于SIMT线程访问全局内存时的数据缓存，Data Cache的空间可配置范围在32KB到128KB，实际内存大小受用户配置的静态和动态内存大小影响，简单计算公式为DataCache空间大小 = UB大小（256KB） - 静态内存 - 动态内存 - 预留空间（8KB）。用户需要合理配置静态和动态内存大小，以确保Data Cache大于或等于32KB。

静态内存分配、动态内存的动态数组分配方式目前开发中，将在后续版本中支持，请关注后续版本。

- 若DataCache小于32KB，会出现校验报错。
- SIMT场景，算子开发不能使用全部的Unified Buffer空间，除了预留8KB空间外，还需至少为SIMT预留32KB的Data Cache空间。

#### 寄存器

SIMT编程中，每个线程拥有独立的寄存器，其生命周期与核函数一致。寄存器的使用由编译器管理，并在核函数执行期间用于线程的局部存储。线程执行时可用寄存器数量与用户配置的blockDim有关，详见下表。

| blockDim大小 | 每个Thread可用寄存器个数(个) |
|--------------|------------------------------|
| 1025~2048 | 16 |
| 513~1024 | 32 |
| 257~512 | 64 |
| 1~256 | 127 |

从上表可知，每个线程块中的线程数量越多，每个线程可使用的寄存器数量就越少。如果用户设置的线程数量过大，而每个线程的计算复杂度又较高时，编译器由于缺乏足够的寄存器来存储本地变量，可能会将数据临时存放到堆栈空间，从而极易导致寄存器溢出（stack spill），影响算子性能。因此，用户应根据实际的算子复杂度，合理配置blockDim。

---

## 4. 核函数

#### 核函数的定义

Ascend C支持开发者自定义核函数来扩展C++，核函数在AI处理器上执行时实际是若干线程在并行执行，每个线程有独立的寄存器和栈，共同完成数据计算任务。

核函数的定义示例如下：

```cpp
__global__ void kernel_name(argument list)
```

定义核函数时需要遵循以下规则：

- 使用函数类型限定符__global__， 标识它是一个核函数。
- 核函数必须具有void返回类型。
- 函数入参支持的类型如下：
    - 基础数据类型，如int32_t、float等。
    - 基础数据类型的指针类型，如int32_t*、float*等，实际上这些指针指向的是GlobalMemory内存。

#### 核函数的调用

算子程序中的函数可以分为三类：host侧执行函数、核函数（device侧执行）、device侧执行函数（除核函数之外）。下图以Kernel直调算子开发方式为例，描述三者的调用关系：

- host侧执行函数可以调用其它host执行函数，也就是通用C/C++编程中的函数调用；也可以通过<<<...>>>调用核函数。
- 核函数可以调用除核函数之外的其它device侧执行函数。
- device侧执行函数（除核函数之外）使用类型限定符__aicore__标识，可以调用同类的其它device侧执行函数。

<img src="./images/kernel_function_call_relationship.png" alt="kernel_function_call_relationship"  width="700px" >

Host侧通过内核调用符<<<...>>>的语法形式调用核函数，如下所示：

```cpp
kernel_name<<<numBlocks, threadsPerBlock, dynUBufSize, stream>>>(args...)
```

执行配置由4个参数决定：

- numBlocks：为核函数配置的线程块的个数，即启用的核数, 支持int32_t和dim3类型；
- threadsPerBlock：每个线程块内并发执行的线程数量，支持int32_t和dim3类型；
- dynUBufSize：动态申请内存空间总大小，一般情况设置为0；
- stream：用于host侧和device侧的流同步。

---

## 5. 同步机制

SIMT是一种单指令多线程的编程模式，其异步编程模型旨在通过多线程并发执行达到内存操作加速的目的。在这一编程模型中，线程作为执行计算或操作内存的最小抽象单位，其操作是相互独立的。然而，在某些应用场景中，需要支持线程间的同步，或防止不同线程对同一内存区域的读写操作引发的数据竞争。为此，Ascend C提供了相应的同步接口，这些接口允许开发者根据需求选择合适的同步机制，以确保异步操作的正确性和性能。

| 接口名 | 功能说明 |
|-------|---------|
| asc_syncthreads | 等待当前线程块内所有线程代码都执行到该函数位置。 |
| asc_threadfence | 用于保证不同线程对同一份全局、共享内存的访问过程中，写入操作的时序性。它不会阻塞线程，仅保证内存操作的可见性顺序。 |
| asc_threadfence_block | 用于协调同一线程块（Thread Block）内线程之间的内存操作顺序，确保某一线程在asc_threadfence_block()之前的所有内存操作（读写），对同一线程块内的其他线程是可见的。 |

---

## 小结

本节介绍了 AI Core SIMT 编程的核心概念：

1. **概述**：AI处理器提供SIMD和SIMT两种编程模型；SIMT适用于离散访问和复杂分支控制场景
2. **抽象硬件架构**：Vector Core包含计算单元、Share Memory、寄存器和栈；Data Cache最大128KB，最小32KB
3. **线程架构**：Grid（gridDim≤65535）→ Thread Block（blockDim≤2048）→ Warp（32线程）；每个AIV核包含4个Warp调度器，scheduler id = warp id % 4
4. **内存层级**：全局内存、共享内存、栈、寄存器；UB 256KB划分为静态内存、动态内存、预留空间（8KB）、Data Cache（32KB~128KB）
5. **核函数**：使用__global__修饰符定义；通过<<<>>>语法调用，配置numBlocks、threadsPerBlock、dynUBufSize、stream参数
6. **同步机制**：提供asc_syncthreads、asc_threadfence、asc_threadfence_block三种同步接口

---

## 课后练习

完成以下题目，检验你对 SIMT 编程模型的理解：

1. （判断题）SIMT 编程模型允许指令对数据进行独立寻址，支持线程级并行计算。    

2. （判断题）在 SIMT 编程中，每个线程拥有独立的寄存器和栈，线程数量越多，每个线程可用的寄存器数量越多。    

3. （单选题）SIMT 线程架构中，一个线程块最多可以启用多少个线程？  
    A. 1024  
    B. 2048  
    C. 4096  
    D. 65535  

4. （单选题）每个 Warp 包含多少个线程？  
    A. 16  
    B. 32  
    C. 64  
    D. 128  

5. （单选题）Unified Buffer 中 Data Cache 的最小容量是多少？  
    A. 8KB  
    B. 16KB  
    C. 32KB  
    D. 128KB  

**执行以下代码获取答案。**

---

上一节：[1.4 regbase 编程基础](./01.04_simd_regbase.ipynb)

In [ ]:
!cat ./answer/01.05_answer.txt